# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply univariate, multivariate, and neighbor-based imputation to incomplete data. 
- Use missingness indicators and understand estimators with native missing-value support. 
- Transform classification and regression targets using scikit-learn target utilities. 


## **1. Imputation Methods**

### **1.1. Basic Value Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_01.jpg?v=1783953340" width="250">



>* Fill missing values with simple substitutes
>* Treat each feature independently

>* Simple, fast, and preserves limited data
>* Can reduce variability and distort categories

>* Respect meaningful missingness with explicit values
>* Fit imputers on training data only



In [ ]:
#@title Python Code - Basic Value Imputation

# Basic imputation fills missing feature values.
# Each column uses its own observed values.
# This example avoids machine learning libraries.

import numpy as np
import pandas as pd

# Create a tiny mixed dataset.
data = {
    "age_years": [29, np.nan, 41, 35, np.nan],
    "height_cm": [170, 165, np.nan, 180, 175],
    "contact": ["email", None, "phone", "email", None],
}

# Build a small dataframe.
df = pd.DataFrame(data)

# Compute numeric replacement values.
age_median = df["age_years"].median()
height_mean = df["height_cm"].mean()

# Compute categorical replacement value.
contact_mode = df["contact"].mode(dropna=True).iloc[0]

# Fill missing values column by column.
filled = df.copy()
filled["age_years"] = filled["age_years"].fillna(age_median)
filled["height_cm"] = filled["height_cm"].fillna(height_mean)
filled["contact"] = filled["contact"].fillna(contact_mode)

# Count missing values before and after.
before_missing = int(df.isna().sum().sum())
after_missing = int(filled.isna().sum().sum())

# Show compact teaching results.
print("Original missing cells:", before_missing)
print("Filled missing cells:", after_missing)
print("Age median used:", age_median)
print("Height mean used:", round(height_mean, 1))
print("Contact mode used:", contact_mode)
print(filled.head().to_string(index=False))



### **1.2. Iterative Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_02.jpg?v=1783953342" width="250">



>* Predict missing values using other features
>* Repeat updates to match dataset patterns

>* Captures relationships across connected features
>* Reliability depends on strong predictive patterns

>* Choose estimators and iterations carefully
>* Fit on training data; report uncertainty



### **1.3. Nearest Neighbor Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_03.jpg?v=1783953344" width="250">



>* Fill gaps using similar records
>* Comparable cases give context-aware estimates

>* Preserves subgroup patterns better than global averages
>* Choose similarity measures carefully across feature types

>* Similarity quality drives imputation reliability
>* Balance neighbor count, cost, and bias



In [ ]:
#@title Python Code - Nearest Neighbor Imputation

# Demonstrate nearest neighbor imputation simply.
# Use tiny numeric patient records.
# Compare local estimates with means.

import math
import pandas as pd

# Create a tiny incomplete dataset.
data = pd.DataFrame(
    {
        "age_years": [25, 27, 52, 55, 29],
        "height_cm": [170, 172, 160, 158, 171],
        "weight_kg": [70, 72, 82, 85, None],
    }
)

# Separate complete and incomplete rows.
known_rows = data[data["weight_kg"].notna()].copy()
missing_row = data[data["weight_kg"].isna()].iloc[0]

# Validate the tiny teaching dataset.
assert len(known_rows) >= 2 and len(data) == 5
assert missing_row[["age_years", "height_cm"]].notna().all()

# Define distance using available features.
def distance_to_missing(row):
    age_gap = row["age_years"] - missing_row["age_years"]
    height_gap = row["height_cm"] - missing_row["height_cm"]
    return math.sqrt(age_gap**2 + height_gap**2)

# Find the two nearest complete records.
known_rows["distance"] = known_rows.apply(distance_to_missing, axis=1)
neighbors = known_rows.nsmallest(2, "distance")

# Average neighbor weights for imputation.
knn_value = neighbors["weight_kg"].mean()
mean_value = known_rows["weight_kg"].mean()

# Fill the missing value using neighbors.
imputed_data = data.copy()
imputed_data.loc[imputed_data["weight_kg"].isna(), "weight_kg"] = knn_value

# Print compact teaching results.
print("Original missing weight row:")
print(data.tail(1).to_string(index=False))
print("Nearest neighbor weights:", neighbors["weight_kg"].tolist())
print("KNN imputed weight:", round(knn_value, 1), "kg")
print("Overall mean weight:", round(mean_value, 1), "kg")
print("Completed final row:")
print(imputed_data.tail(1).to_string(index=False))



## **2. Missingness Signals**

### **2.1. Missingness Indicators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_01.jpg?v=1783953348" width="250">



>* Flags values that were originally missing
>* Preserves missingness as useful model signal

>* Missingness can reflect real-world processes
>* Indicators distinguish imputed from observed values

>* Indicators can add noise and complexity
>* Evaluate missingness as a useful signal



### **2.2. Empty Feature Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_02.jpg?v=1783953346" width="250">



>* Features can be entirely empty during fitting
>* Imputers cannot estimate values without observations

>* Choose dropping, keeping, or signaling emptiness
>* Placeholders preserve structure but need caution

>* Indicators separate imputed from observed values
>* Interpret missingness using real-world context



In [ ]:
#@title Python Code - Empty Feature Handling

# Empty features need careful handling.
# Indicators preserve useful missingness signals.
# This example avoids external libraries.

# Store tiny training rows as dictionaries.
train_rows = [
    {"age": 34, "lab_score": None, "income": 52000},
    {"age": 41, "lab_score": None, "income": 61000},
    {"age": 29, "lab_score": None, "income": 48000},
]

# Store future rows with observed values.
future_rows = [
    {"age": 37, "lab_score": 8.2, "income": 57000},
    {"age": 45, "lab_score": None, "income": 69000},
]

# Choose stable feature order explicitly.
features = ["age", "lab_score", "income"]

# Compute means from observed training values.
means = {}
for feature in features:
    values = [row[feature] for row in train_rows if row[feature] is not None]
    means[feature] = sum(values) / len(values) if values else 0.0

# Transform rows with placeholders and indicators.
def transform_rows(rows, features, means):
    transformed = []
    for row in rows:
        new_row = {}
        for feature in features:
            missing = row[feature] is None
            new_row[feature] = means[feature] if missing else row[feature]
            new_row[feature + "_missing"] = int(missing)
        transformed.append(new_row)
    return transformed

# Apply the same structure to both datasets.
train_ready = transform_rows(train_rows, features, means)
future_ready = transform_rows(future_rows, features, means)

# Summarize the empty feature decision.
empty_features = [name for name in features if all(row[name] is None for row in train_rows)]
print("Empty during fitting:", empty_features)
print("Placeholder for lab_score:", means["lab_score"])
print("Training columns kept:", list(train_ready[0].keys()))
print("First transformed training row:", train_ready[0])
print("First transformed future row:", future_ready[0])



### **2.3. Native Missing Support**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_03.jpg?v=1783953350" width="250">



>* Some estimators handle missing values directly
>* Missingness can be useful predictive signal

>* Trees can route missing values during splits
>* Native support preserves missingness as signal

>* Monitor changing missingness patterns during deployment
>* Preprocess when needed; avoid unnecessary imputation



## **3. Target Transformation Tools**

### **3.1. Encoding Class Labels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_01.jpg?v=1783953333" width="250">



>* Convert text class labels into numbers
>* Preserve categories without implying class order

>* Learn label mappings from training classes
>* Convert predictions back to readable labels

>* Target labels are identifiers, not ordered values
>* Encoding supports evaluation and readable results



In [ ]:
#@title Python Code - Encoding Class Labels

# Encode class labels for classification targets.
# Keep mappings stable across workflow stages.
# Decode predictions back to readable labels.

# No extra installations are required.

# Import small built-in tools only.
import numpy as np

# Create a tiny text target vector.
y_text = np.array([
    "low risk", "high risk", "moderate risk",
    "low risk", "high risk", "moderate risk"
])

# Learn sorted unique class names.
classes = np.unique(y_text)

# Build a stable label-to-number mapping.
label_to_id = {label: idx for idx, label in enumerate(classes)}

# Build the reverse number-to-label mapping.
id_to_label = {idx: label for label, idx in label_to_id.items()}

# Encode each original class label.
y_encoded = np.array([label_to_id[label] for label in y_text])

# Pretend these are model predictions.
predicted_ids = np.array([0, 2, 1])

# Decode numeric predictions for people.
predicted_labels = np.array([id_to_label[idx] for idx in predicted_ids])

# Check that encoding preserved sample count.
assert y_encoded.shape[0] == y_text.shape[0]

# Print compact teaching results.
print("Classes:", classes.tolist())
print("Mapping:", label_to_id)
print("Original targets:", y_text[:4].tolist())
print("Encoded targets:", y_encoded[:4].tolist())
print("Decoded predictions:", predicted_labels.tolist())



### **3.2. Multilabel Binarization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_02.jpg?v=1783953338" width="250">



>* Examples can have multiple labels
>* Labels become binary columns for models

>* Converts many labels into yes-or-no columns
>* Standardizes messy targets for model training

>* Preserve label vocabulary and column order
>* Convert predictions back to label sets



In [ ]:
#@title Python Code - Multilabel Binarization

# Multilabel targets can contain several labels.
# Binarization creates one column per label.
# This example uses only built-in Python.

# Define tiny multilabel targets for songs.
song_labels = [
    {"energetic", "happy"},
    {"relaxing", "romantic"},

    {"energetic", "romantic"},
    {"melancholic"},
    set(),
]

# Collect a stable vocabulary from training labels.
label_names = sorted(
    {label for labels in song_labels for label in labels}
)

# Validate that labels were discovered safely.
if not label_names:
    raise ValueError("At least one label is required.")

# Convert each label set into binary columns.
binary_rows = [
    [int(label in labels) for label in label_names]
    for labels in song_labels
]

# Convert binary rows back into readable labels.
recovered_labels = [
    {label for label, flag in zip(label_names, row) if flag}
    for row in binary_rows
]

# Print compact results for classroom inspection.
print("Label columns:", label_names)
print("First song labels:", sorted(song_labels[0]))
print("First binary row:", binary_rows[0])
print("Empty-label song row:", binary_rows[4])
print("Recovered third song:", sorted(recovered_labels[2]))
print("Matrix shape:", len(binary_rows), "rows x", len(label_names), "columns")



### **3.3. Target Transformations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_03.jpg?v=1783953336" width="250">



>* Transform skewed targets before regression modeling
>* Convert predictions back to original units

>* Wrap transformations with regressors for consistency
>* Model targets better without changing meaning

>* Choose transformations that fit target values
>* Evaluate and deploy on original scale



# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>


In this lecture, you learned to:
- Apply univariate, multivariate, and neighbor-based imputation to incomplete data. 
- Use missingness indicators and understand estimators with native missing-value support. 
- Transform classification and regression targets using scikit-learn target utilities. 

In the next Module (Module 6), we will go over 'Composite Estimators'